![Week 5 Day 5](../assets/banners/w5d5.png)

# Week 5 Day 5 - 🧱 Grader vs Human: Golden Set, MAE, QWK, CER/WER and Regression Tests

Yesterday GradeWise ran end to end: answer sheets in, report cards out, with Emma in the loop whenever something was flagged. It *works*.

But "it works" isn't the question Emma will ask. She'll ask: **"Would it give the same marks I would?"** And so will her head of department, the students, and their parents.

Today we answer with numbers. We compare GradeWise's marks against a human examiner's on a **golden set**, using three metrics that each tell a different story: **MAE**, **exact agreement** and **quadratic weighted kappa**. You'll work them out by hand on a tiny example first, so they never feel like magic. Then we measure the transcription side with **CER and WER**, lock the results in with **regression tests**, and wrap up the week.

![Week 5 roadmap](../assets/roadmap/week5.png)

### 🎯 By the end of this lab you will
1. 🥇 Know what a golden set is, and load GradeWise's (150 human-marked answers)
2. 🧮 Compute MAE, exact agreement, within-1 and **QWK by hand**, and know when each one lies
3. 🧱 Measure the stand-in grader: a QWK of about **0.85, the baseline your real grader must beat**
4. ✍️ Score transcriptions with CER and WER
5. 🧪 Turn quality into **regression tests** with `pytest`

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | 🧱 Day 5 - Grader vs Human: Golden Set, MAE and Quadratic Weighted Kappa | 🧪 |
| 2 | Day 5 - Transcription CER/WER and Regression Tests with pytest | 🧪 |
| 3 | Day 5 - Week 5 Wrap-Up: Summary, Roadmap and Quiz | 🎬 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Almost everything today is <b>free</b>: the metrics are plain maths, and the baseline comes from the 🎭 stand-in grader (<code>KeywordGraderModel</code>). The one 🔑 cell grades a few students with your real model so you can compare it with the baseline (with `gpt-6-luna`, about a cent per student; $0 but rate-limited on `MODEL_MODE=free`). Results go to a sandbox, <code>output/week5/day5/</code>.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json

from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise import datasets
from gradewise.config import DATA_DIR, OUTPUT_DIR, REPO_ROOT, settings
from gradewise.console import success, tip
from gradewise.evals.compat import import_pandas
from gradewise.ingestion import ingest
from gradewise.rag import get_retriever
from gradewise.schemas import AnswerSheet, ExamPaper, TranscribedAnswer

pd = import_pandas()  # 💡 = `import pandas as pd`, with a fix for locked-down Windows machines

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
FULL_EXAM = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTIONS = {q.id: q for q in FULL_EXAM.questions}  # every question, for the one-answer demos
TRUTH = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))

# 📦 What the batch runs grade: all 15 parts and 10 students, or Q1–Q3 and 2 students with DATA_SIZE=small
exam = datasets.subset_exam(FULL_EXAM)
STUDENTS = datasets.synthetic_students()

# 🆓 A keyword-only (BM25) copy of the CS201 notes: fake embeddings, real keyword search, zero cost
OFFLINE_DB = OUTPUT_DIR / "week5_offline_db"
fake_embeddings = DeterministicFakeEmbedding(size=64)
ingest(
    collection="cs201_offline",
    embeddings=fake_embeddings,
    persist_dir=OFFLINE_DB,
    quiet=True,
    files=datasets.kb_files(),
)  # 📦 all 4 PDFs, or unit1 + the handbook with DATA_SIZE=small
bm25 = get_retriever("cs201_offline", k=4, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)

success(
    "Ready",
    {
        "Exam": f"{exam.title} ({len(exam.questions)} questions, {exam.total_marks:g} marks)",
        "Students": f"{len(STUDENTS)} synthetic scripts",
        "Offline KB": f"cs201_offline (BM25, {len(datasets.kb_files())} PDFs)",
        "Grading model": settings.grading_model,
    },
)

## Part 1: The golden set

A **golden set** is a collection of answers marked by a trusted human, used as the answer key for your system. GradeWise's lives in `data/golden/synthetic_labels.csv`: a human examiner marked all ten synthetic scripts with the official rubric, **criterion by criterion**.

📦 `datasets.golden_rows()` keeps every row, or only the 2 students × Q1–Q3 when `DATA_SIZE=small`.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>Small data, noisy metrics.</b> With <code>DATA_SIZE=small</code> the golden slice is just 8 answers, so every number below (MAE, exact agreement, QWK, the confusion counts and the biggest gaps) will differ from the text and can swing wildly from one tiny change. That's fine for checking that everything <i>runs</i>, never for judging a grader. The figures quoted in the text come from the full set (<code>DATA_SIZE=full</code>).</span>
        </td>
    </tr>
</table>

In [ ]:
from gradewise.evals.grading_eval import GOLDEN_CSV, human_marks, load_golden

golden = datasets.golden_rows(load_golden())  # 📦 all 150 answers, or 2 students × Q1–Q3
print(
    f"{GOLDEN_CSV.relative_to(REPO_ROOT)}: {len(golden)} criterion marks, "
    f"{golden.student_id.nunique()} students × {golden.question_id.nunique()} questions"
)
golden.head(8)

In [ ]:
human, max_marks = human_marks(golden)  # 💡 summed per (student, question)
print(f"{len(human)} answers. Sam's Q3: {human[('student_05', 'Q3')]:g}/{max_marks['Q3']:g}")
if ("student_10", "Q8") in human:  # 💡 not in the DATA_SIZE=small slice
    print(f"student_10's Q8: {human[('student_10', 'Q8')]:g}/{max_marks['Q8']:g}")

Four rules make a golden set worth trusting:

1. **Marked independently** of the system you're testing (never "correct the AI's marks": that anchors the human).
2. **Same rubric, same principles** as the grader: otherwise you're measuring a disagreement about the rules, not about the answers.
3. **Covers the range**: strong, weak, blank, misspelt, off-topic, and adversarial answers (student 10!).
4. **Never tuned on.** If you edit the prompt until the golden set looks good, you've overfitted to 150 answers. Keep some back, or collect more.

💡 Criterion-level labels let you see *where* a disagreement comes from: "it misses the definition marks" is fixable; "it disagrees by 1.3 marks on average" isn't.

## Part 2: Three metrics, worked out by hand

Here's a tiny example: six answers to a 4-mark question, marked by a human and by two graders, A and B.

In [ ]:
import numpy as np

h = np.array([4, 3, 2, 0, 1, 3])  # 🧑‍🏫 the human
a = np.array([4, 2, 2, 1, 1, 4])  # 🤖 grader A
b = np.array([4, 0, 2, 4, 1, 0])  # 🤖 grader B

pd.DataFrame(
    {
        "answer": range(1, 7),
        "human": h,
        "grader A": a,
        "|A − human|": abs(a - h),
        "grader B": b,
        "|B − human|": abs(b - h),
    }
).set_index("answer")

### 1. Exact agreement: "how often is it *exactly* right?"
Count the matches. A matches on answers 1, 3 and 5; so does B. **Both 3/6 = 50%.**

### 2. MAE (mean absolute error): "how far off is it, on average, in marks?"
Average the gaps. A: (0+1+0+1+0+1)/6 = **0.5 marks**. B: (0+3+0+4+0+3)/6 = **1.67 marks**.

Exact agreement says A and B are equally good. MAE says B is more than three times worse. MAE is easy to explain to a teacher ("off by half a mark on average"), but it has blind spots too: it doesn't care whether errors are spread evenly or concentrated, and it ignores what agreement you'd get **by chance** (a grader that gives everyone 2/4 can have a decent MAE).

In [ ]:
from gradewise.evals import exact_agreement, mae, within

for name, g in [("A", a), ("B", b)]:
    print(
        f"grader {name}: exact {exact_agreement(h, g):.0%} · MAE {mae(h, g):.2f} · within 1 mark {within(h, g, 1):.0%}"
    )

### 3. Quadratic weighted kappa (QWK): "how much better than chance, with big misses punished hardest?"

QWK is the standard metric for automated essay and short-answer scoring. It combines two ideas:
- **Kappa** compares the agreement you observed with the agreement you'd expect **by chance** from each rater's mark distribution. 1 = perfect, 0 = no better than chance, below 0 = worse than chance.
- **Quadratic weights**: a disagreement of *d* marks costs *d²*. Off by 1 costs 1, off by 3 costs 9. That's how a moderator thinks: a one-mark difference is a judgement call, a three-mark difference is a mistake.

Four small steps. **Step 1:** the *observed* matrix O: how often the human gave mark *i* and the grader gave *j*.

In [ ]:
K = 5  # possible marks 0, 1, 2, 3, 4
O = np.zeros((K, K))  # noqa: E741 (O and E, as in the formula)
for i, j in zip(h, a, strict=True):
    O[i, j] += 1
pd.DataFrame(O.astype(int), index=[f"human {m}" for m in range(K)], columns=[f"A {m}" for m in range(K)])

**Step 2:** the *weights* W: the cost of each cell, (i − j)² scaled to 0…1. The diagonal (agreement) is free.

**Step 3:** the *expected* matrix E: what O would look like if the two raters marked independently, each keeping their own mark distribution. It's the outer product of the row and column totals, divided by N.

In [ ]:
W = np.array([[(i - j) ** 2 / (K - 1) ** 2 for j in range(K)] for i in range(K)])
E = np.outer(O.sum(axis=1), O.sum(axis=0)) / O.sum()
print("W =\n", W.round(3), "\n\nE =\n", E.round(2))

**Step 4:** kappa = 1 − (weighted disagreement observed) / (weighted disagreement expected by chance):

$$\kappa = 1 - \frac{\sum_{ij} W_{ij} O_{ij}}{\sum_{ij} W_{ij} E_{ij}}$$

In [ ]:
from gradewise.evals import quadratic_weighted_kappa


def qwk_by_hand(human_marks, grader_marks, k=K) -> float:
    O = np.zeros((k, k))  # noqa: E741
    for i, j in zip(human_marks, grader_marks, strict=True):
        O[i, j] += 1
    E = np.outer(O.sum(axis=1), O.sum(axis=0)) / O.sum()
    return 1 - (W * O).sum() / (W * E).sum()


for name, g in [("A", a), ("B", b)]:
    print(f"grader {name}: QWK by hand {qwk_by_hand(h, g):.3f} · gradewise {quadratic_weighted_kappa(h, g):.3f}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Grader A and grader B have <b>the same exact agreement (50%)</b>. Yet A's QWK is high and B's is <i>negative</i>: worse than marking at random. Before reading on, explain why in one sentence. (Hint: look at <i>how far</i> each grader's misses are, and remember a miss of 3 costs 9 times as much as a miss of 1.)</span>
        </td>
    </tr>
</table>

So, which metric should you report? **All three, plus within-1**, because each one answers a different question:

| Metric | Answers | Misleads when… |
|---|---|---|
| **Exact agreement** | "How often is the mark exactly right?" | most misses are small (it can't tell 1 from 4) |
| **Within 1 mark** | "How often is it close enough not to argue about?" | the question is only worth 1–2 marks |
| **MAE** | "How many marks off, on average?" | marks bunch around the middle (chance looks decent) |
| **QWK** | "How much better than chance, punishing big misses?" | everyone gets the same mark (undefined), or very few answers |

A rough reading guide for kappa (Landis & Koch): below 0.4 weak, 0.4–0.6 moderate, 0.6–0.8 substantial, **above 0.8 strong**. Two trained human examiners don't agree perfectly either, so their agreement with each other is the realistic ceiling for any grader.

⚠️ Two details hidden in `gradewise/evals/grading_eval.py`: QWK needs whole-number labels, so half marks are **rounded half up** with `round_marks()` (Python's `round(2.5)` is 2: banker's rounding would bias the kappa); and when both raters give everyone the same single mark, kappa is undefined, so we return 1.0 if they agree and 0.0 otherwise.

In [ ]:
from gradewise.evals.grading_eval import round_marks

print("round()       :", [round(x) for x in (0.5, 1.5, 2.5, 3.5)])
print("round_marks() :", round_marks([0.5, 1.5, 2.5, 3.5]))

## Part 3: 🧱 The baseline your real grader must beat

Time for the real golden set. We grade the class (`STUDENTS`: all ten, or two with `DATA_SIZE=small`) with the 🎭 stand-in through the **real orchestrator** (Day 4), from the ground-truth transcripts so only the grader is being tested, with `auto_approve=True` because an evaluation must measure the grader's **own** marks, untouched by a reviewer. That's exactly what `week5/evaluation/run_evals.py --fake` does.

In [ ]:
from datetime import datetime

from gradewise.grading.fakes import KeywordGraderModel
from gradewise.orchestrator import grade_batch, sheets_from_transcripts

SANDBOX = OUTPUT_DIR / "week5" / "day5"
EVAL_BUILD = {"feedback": "rules", "pdf": False, "index": False, "checkpointer": "memory", "retriever": bm25}

baseline_run = grade_batch(
    exam.exam_id,
    sheets_from_transcripts(exam, students=STUDENTS),
    thread_id=f"day5-baseline-{datetime.now():%H%M%S}",
    exam=exam,
    auto_approve=True,
    llm=KeywordGraderModel(),
    db=SANDBOX / "baseline.db",
    reports_dir=SANDBOX / "reports",
    **EVAL_BUILD,
)  # 🎭 stand-in grader
print(f"{baseline_run.status}: {len(baseline_run.reports)} students graded")

In [ ]:
from gradewise.evals import evaluate_grading

baseline = evaluate_grading(baseline_run.reports, show=True, save=False)
BASELINE_QWK = baseline["overall"]["qwk"]

### 📏 The baseline: QWK ≈ 0.85

There it is: the keyword stand-in agrees with the human examiner with a **QWK of about 0.85**, an MAE of about half a mark, and 91% of answers within one mark. That sounds impressive for something that can't read! It's because the synthetic answers use the rubric's own vocabulary quite often, and QWK rewards getting the *ordering* of strong and weak answers right.

So this is **the baseline your real grader must beat**. If an LLM grader, with all its cost, can't beat a keyword matcher on QWK, MAE and exact agreement, it isn't earning its keep. And look *where* the stand-in fails: a confusion summary shows whether a grader is too harsh or too generous.

In [ ]:
c = baseline["confusion"]
print(
    f"same mark {c['agree']} · grader higher {c['grader_higher']} · grader lower {c['grader_lower']} · "
    f"mean signed error {c['mean_signed_error']:+.2f} marks"
)
pd.DataFrame(baseline["largest_gaps"])

Negative mean signed error: the stand-in is **harsh** overall (73 answers marked lower than the human, 16 higher). Let's read the student's answer behind the single biggest gap:

In [ ]:
gap = baseline["largest_gaps"][0]
q = QUESTIONS[gap["question_id"]]
print(f"{gap['student_id']} · {q.number} ({q.max_marks:g} marks): human {gap['human']:g}, stand-in {gap['grader']:g}\n")
print(TRUTH[gap["student_id"]].get(gap["question_id"], "(blank)"))

Student 04 made **one slip** (8 KB is 2¹³ bytes, so the offset is 13 bits, not 12) and then carried it through every later step *correctly*. The human examiner applied the handbook's principles 5 and 6 (method marks, and follow-through: *error carried forward*) and gave 4/5. The stand-in looks for "13 bits", "2^19" and "2 MB", finds none of them, and gives 1.5. Further down the list you'll find the opposite failures too: answers that say the right thing in different words (principle 3), and keyword-rich answers that don't actually explain anything (principle 4). Judging **method and meaning** is exactly what you're paying an LLM grader for.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">A baseline is a <b>floor</b>, not a target. Beat it on <i>every</i> metric, and then check the per-question table: an LLM grader that's brilliant overall but much worse than the stand-in on the numerical questions (Q5, Q7a, Q7b) still has a problem worth fixing. And never tune your prompt on the same 150 answers you report: that's marking your own homework.</span>
        </td>
    </tr>
</table>

## Part 4: 🔑 Your real grader vs the baseline

Grade a few students with your real grading model and compare. In the default `MODEL_MODE=cheap` that's `gpt-6-luna` with a higher reasoning effort: three students (45 answers) cost a few cents, and `STUDENTS_FOR_REAL_EVAL = STUDENTS` grades the whole golden set for well under a dollar. `MODEL_MODE=free` costs $0, but OpenRouter's free models allow 20 requests a minute and 50 a day, so keep it to `DATA_SIZE=small` (GradeWise retries `429`s and drops to two parallel calls). And since a different free model may mark each answer, its QWK and accuracy vary from run to run: never compare a free run with a cheap one as if they were the same grader. Want a stronger grader to compare? `MODEL_MODE=custom` with `SMART_MODEL=openrouter:openai/gpt-6.1-terra` ($1 / $5 per million tokens, about ten times the price). The metrics only use the (student, question) pairs both sides have, so a partial run is still a fair comparison on those students.

In [ ]:
from gradewise.errors import GradeWiseError

STUDENTS_FOR_REAL_EVAL = STUDENTS[:3]  # 📦 3 students (2 with DATA_SIZE=small)

try:
    real_run = grade_batch(
        exam.exam_id,
        sheets_from_transcripts(exam, students=STUDENTS_FOR_REAL_EVAL),
        thread_id=f"day5-real-{datetime.now():%H%M%S}",
        exam=exam,
        auto_approve=True,
        db=SANDBOX / "real.db",
        reports_dir=SANDBOX / "real_reports",
        **EVAL_BUILD,
    )
    real = evaluate_grading(real_run.reports, show=True, save=True)
    ids = set(real["students"])
    same_students = evaluate_grading([r for r in baseline_run.reports if r.student_id in ids], show=False, save=False)
    display(pd.DataFrame({"🎭 stand-in": same_students["overall"], f"🤖 {settings.grading_model}": real["overall"]}))
except GradeWiseError as err:
    err.show()

## Part 5: Transcription: CER and WER

Grading is only half of GradeWise's accuracy. If the vision model misreads Sam, even a perfect grader marks the wrong answer. Week 4 gave us the two standard measures, both edit distances against the ground truth:

- **CER** (character error rate) = character insertions + deletions + substitutions, divided by the reference length
- **WER** (word error rate) = the same, counted in words

`gradewise/evals/transcription_eval.py` runs them over whole sheets. Three pretend "vision models" on Sam's script show what each metric notices:

In [ ]:
import re

from gradewise.evals import evaluate_transcriptions

sam_truth = {"student_05": TRUTH["student_05"]}


def sheet_from(texts: dict[str, str], confidence: float = 0.95) -> AnswerSheet:
    answers = [
        TranscribedAnswer(question_id=q.id, text=texts.get(q.id, ""), confidence=confidence) for q in exam.questions
    ]
    return AnswerSheet(
        student_id="student_05", exam_id=exam.exam_id, source_file="student_05.pdf", page_count=3, answers=answers
    )


def tidy(text: str) -> str:  # "helpfully" capitalises and adds full stops: week 4's cardinal sin
    return re.sub(r"(^|\n)(\w)", lambda m: m[1] + m[2].upper(), text).replace("\n", ".\n") + "."


readers = {
    "perfect": sheet_from(TRUTH["student_05"]),
    "tidies up": sheet_from({k: tidy(v) for k, v in TRUTH["student_05"].items()}),
    "skips line 2": sheet_from({k: v.split("\n")[0] for k, v in TRUTH["student_05"].items()}),
}
pd.DataFrame({name: evaluate_transcriptions([s], sam_truth)["overall"] for name, s in readers.items()}).T[
    ["n", "cer", "wer"]
]

"Tidying up" looks harmless, yet look at its WER: **higher than skipping a whole line**, because every capitalised or punctuated word counts as a wrong word. CER tells the other half of the story (only a few characters changed, while the skipped lines lose a quarter of them). That's why we report both, and the point stands either way: CER and WER measure **faithfulness**, and a transcriber that edits the student is unfaithful.

If you've transcribed real sheets (week 4's `transcription_graph.py` saves them in `output/transcriptions/`), `run_transcription_eval()` scores them all and also checks the **confidence calibration**: low-confidence answers *should* have a higher CER, or the review threshold is flagging the wrong answers.

In [ ]:
from gradewise.evals import run_transcription_eval

try:
    transcription = run_transcription_eval(save=False)
except GradeWiseError as err:
    tip(f"No transcriptions yet ({err.what}). Run week 4's transcription_graph.py on a sheet, then re-run this cell.")

## Part 6: Regression tests with `pytest`

Every number above can quietly get worse: you tweak the examiner prompt, upgrade LangChain, switch `MODEL_MODE` or `SMART_MODEL`, or a teammate "simplifies" `round_marks()`. A **regression test** turns "it was good last week" into a check that runs on every change. GradeWise uses two kinds:

| Kind | Example | Runs |
|---|---|---|
| **Unit tests of the maths** | `test_qwk_matches_a_hand_computation` in `tests/test_evals.py` | every commit: free, fast, deterministic |
| **Quality gates** | "the grader's QWK on the golden set is at least the baseline" | with the stand-in: every commit. With a real model: nightly or before a release (it costs money, and isn't fully deterministic, so allow a small tolerance) |

Here's one of the unit tests, a hand computation just like Part 2:

In [ ]:
import ast

source = (REPO_ROOT / "tests" / "test_evals.py").read_text(encoding="utf-8")
test = next(n for n in ast.parse(source).body if getattr(n, "name", "") == "test_qwk_matches_a_hand_computation")
print(ast.get_source_segment(source, test))

Now a **quality gate** of our own. The next cell writes a small test file into the sandbox and runs it with `pytest`, exactly as CI would. It grades the golden set with the free stand-in and fails if agreement ever drops below the baseline, or if the planted injection stops being flagged. (When your real grader beats the baseline, copy the idea and raise the bar.) 💡 A gate must be stable, so it always checks the **full** golden set (`DATA_SIZE=full` for the test run): it's free, and 8 answers would be far too noisy to gate on.

In [ ]:
import subprocess
import sys

GATE = SANDBOX / "test_quality_gate.py"
GATE.parent.mkdir(parents=True, exist_ok=True)
GATE.write_text(
    """
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

from gradewise.config import DATA_DIR
from gradewise.evals import evaluate_grading
from gradewise.grading import build_grading_graph, grade_sheet
from gradewise.grading.fakes import KeywordGraderModel
from gradewise.orchestrator import sheets_from_transcripts
from gradewise.schemas import ExamPaper

BASELINE_QWK = 0.80  # 💡 a little below the measured 0.85, so harmless noise doesn't fail the build


class OneDoc(BaseRetriever):
    def _get_relevant_documents(self, query, *, run_manager=None):
        meta = {"source": "unit1_operating_systems.pdf", "page": 4, "section": "1.6 Deadlocks", "chunk_index": 0}
        text = "A deadlock is a set of processes, each waiting for a resource held by another."
        return [Document(text, metadata=meta)]


def graded_class():
    exam = ExamPaper.model_validate_json((DATA_DIR / "exams/cs_midterm/expected_exam.json").read_text(encoding="utf-8"))
    graph = build_grading_graph(KeywordGraderModel(), OneDoc())
    return [grade_sheet(exam, sheet, graph=graph) for sheet in sheets_from_transcripts(exam)]


def test_agreement_with_the_human_examiner_meets_the_baseline():
    overall = evaluate_grading(graded_class(), show=False, save=False)["overall"]
    assert overall["qwk"] >= BASELINE_QWK, overall
    assert overall["within_1"] >= 0.85, overall


def test_the_planted_injection_is_always_flagged():
    flagged = {(r.student_id, g.question_id) for r in graded_class() for g in r.results if g.needs_review}
    assert ("student_10", "Q8") in flagged
""",
    encoding="utf-8",
)

run = subprocess.run(
    [sys.executable, "-m", "pytest", str(GATE), "-q", "-p", "no:cacheprovider", "-p", "no:warnings"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    env={**os.environ, "PYTHONIOENCODING": "utf-8", "DATA_SIZE": "full"},
)  # 💡 gate on the full set
print(run.stdout[-1500:] or run.stderr[-1500:])

And the week's real test suites, all free. (In a terminal: `uv run python -m pytest tests/test_evals.py tests/test_grading.py -q`.)

In [ ]:
suites = ["tests/test_evals.py", "tests/test_grading.py", "tests/test_reports.py", "tests/test_orchestrator.py"]
run = subprocess.run(
    [sys.executable, "-m", "pytest", *suites, "-q", "-p", "no:cacheprovider", "-p", "no:warnings"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"},
)
print(run.stdout.strip().splitlines()[-1] if run.stdout.strip() else run.stderr[-1500:])

## Part 7: 🚀 Run the milestone

This week's evaluation script is `week5/evaluation/run_evals.py`. Open it: `grade_synthetic()` grades the synthetic students (all ten, or Q1–Q3 for two with `DATA_SIZE=small`) from the transcripts through `grade_batch()`, then `run_grading_eval()` and `run_transcription_eval()` print the tables you saw above and save them to `output/evals/`.

The free version, with the stand-in grader (about ten seconds):

In [ ]:
run = subprocess.run(
    [sys.executable, "week5/evaluation/run_evals.py", "--fake", "--only", "grading"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    env={**os.environ, "PYTHONIOENCODING": "utf-8", "COLUMNS": "100"},
)
print(run.stdout[-3500:] or run.stderr[-2000:])

And with your real grader, in a terminal (all ten students: well under a dollar with `gpt-6-luna`; a few cents with `DATA_SIZE=small`; $0, more slowly, with `MODEL_MODE=free`):

```bash
uv run week5/evaluation/run_evals.py --grade          # grade the golden set with your grading model, then evaluate
uv run week5/evaluation/run_evals.py                  # evaluate whatever is already stored (e.g. after Day 4)
```

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">🎉 <b>Milestone M5 reached.</b> GradeWise grades, explains and <i>proves</i> itself: parallel, evidence-based grading with guardrails (<code>gradewise/grading/</code>), report cards and a results agent (<code>gradewise/reports/</code>), the end-to-end orchestrator with human review (<code>gradewise/orchestrator.py</code>), and an honest evaluation against human marks (<code>gradewise/evals/</code>). Run the whole thing with <code>uv run week5/project/orchestrator.py</code>; see <code>week5/project/README.md</code> and <code>week5/evaluation/README.md</code>.</span>
        </td>
    </tr>
</table>

## Part 8: Week 5 wrap-up

### What GradeWise learned this week

| Day | You built | Lives in |
|:-:|---|---|
| 1 | An evidence-based grader: rubric + verbatim answer + notes → `GradeResult` | `gradewise/grading/prompts.py`, `grader.py` |
| 2 | Tools via `ToolRuntime`, `Send` per question, guardrails and injection defence | `tools.py`, `graph.py`, `guardrails.py` |
| 3 | Report cards (HTML, PDF), SQLite + Chroma storage, "chat with results" | `gradewise/reports/` |
| 4 | 🧱 The orchestrator: end to end, interrupts, resume, events, LangSmith | `gradewise/orchestrator.py` |
| 5 | 🧱 The golden set, MAE / exact / QWK, CER / WER and regression tests | `gradewise/evals/`, `week5/evaluation/` |

### 🗺️ Where we are heading

![Diagram](../assets/diagrams/ab7922d3bf03.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  W1[W1 Exam → JSON] --> W2[W2 Knowledge base] --> W3[W3 Agents & graphs] --> W4[W4 Evidence + handwriting] --> W5[W5 Grading & reports]:::here --> W6[W6 Ship it]
  classDef here fill:#7c5cff,color:#fff
```

</details>


**Next week** we **ship it**. The orchestrator gets a FastAPI backend (upload, run, status), a Streamlit dashboard for Emma with a **review queue** for exactly the interrupts you resumed by hand on Day 4, live progress from the `on_event` stream, the report-card viewer and "chat with results", all in one `docker compose up`. Then we deploy it, and talk honestly about responsible AI grading.

📝 **Quiz 5** is waiting for you on Udemy (10 questions): metrics, guardrails, `Send`, interrupts and a "what's the bug" or two.

## Recap, and where we are heading

Today you:
- 🥇 loaded the golden set and learned the four rules that make one trustworthy
- 🧮 computed exact agreement, MAE, within-1 and QWK **by hand**, and saw two graders with the same exact agreement and opposite QWKs
- 🧱 measured the stand-in grader: QWK ≈ 0.85, **the baseline your real grader must beat**, and found where it fails (equivalent wording)
- ✍️ scored transcriptions with CER and WER, and saw why "tidying up" is an error
- 🧪 wrote a quality gate with `pytest`, and ran the week's test suites and milestone script

⏭ **Next:** the week 5 exercise below, then `week6/day1.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">This week's big exercise is <code>week5/week5_EXERCISE.ipynb</code>: <b>add a moderation second-grader for borderline marks</b>. Real exam boards don't trust one marker either: a moderator re-marks a sample, and big disagreements go to a senior examiner. You'll decide which marks are "borderline", have a second, independent grader re-mark only those, combine the two (or escalate to Emma when they disagree), and use today's golden-set metrics to prove whether moderation is worth its cost. It runs free with two stand-ins first. Solution: <code>week5/solutions/week5_EXERCISE_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/thankyou.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ec4899;">Thank you</h2>
            <span style="color:#ec4899;">That's week 5 done, and it was the big one: GradeWise can now grade a whole class fairly, explain every mark, take a teacher's decision, and prove how good it is. Thank you for working through it! If something didn't work, the Udemy Q&A is the place to ask, and if you've built something cool on top of this week (a better grader prompt that beats the baseline, perhaps?), share it in <code>community_contributions/</code>. See you in week 6, where we ship it. 🚀</span>
        </td>
    </tr>
</table>